## YOLOv8n Threshold 최적화

셀 1: 라이브러리 import

In [ ]:
# ============================================
# YOLOv8n Threshold 최적화
# ============================================

# YOLO 모델
from ultralytics import YOLO

# 경로 처리
from pathlib import Path

# 이미지 처리
import cv2

# 데이터 처리
import pandas as pd

# JSON
import json

# 시각화
import matplotlib.pyplot as plt

# 한글 폰트
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print("✅ 라이브러리 import 완료")

셀 2: 경로 설정

In [ ]:
# ============================================
# 경로 설정
# ============================================

# 프로젝트 루트
PROJECT_ROOT = Path(r'N:\개인\이수빈\3.13_Mini_Project')

# YOLOv8n 모델 (Best)
MODEL_PATH = PROJECT_ROOT / 'results' / 'yolov8n_70_15' / 'weights' / 'best.pt'

# Test 데이터
TEST_DIR = PROJECT_ROOT / 'DATASET' / 'test_set_894'

# 결과 저장
EVAL_DIR = PROJECT_ROOT / 'evaluation' / 'threshold_optimization'
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("🎯 YOLOv8n Threshold 최적화")
print("=" * 60)
print(f"\n모델: {MODEL_PATH}")
print(f"Test: {TEST_DIR}")
print(f"결과: {EVAL_DIR}")
print("=" * 60)

셀 3: 모델 로드 및 데이터 준비

In [ ]:
# ============================================
# 모델 로드 및 데이터 준비
# ============================================

print("\n[모델 로드]")

# YOLOv8n Best 모델 로드
model = YOLO(str(MODEL_PATH))
print("✅ YOLOv8n 로드 완료")

# Test 이미지 리스트 (화재)
test_fire = [f for f in (TEST_DIR / 'fire').iterdir() 
             if f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']]

# Test 이미지 리스트 (정상)
test_normal = [f for f in (TEST_DIR / 'normal').iterdir() 
               if f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']]

print(f"\n[Test 데이터]")
print(f"  화재:  {len(test_fire)}장")
print(f"  정상:  {len(test_normal)}장")
print(f"  총:    {len(test_fire) + len(test_normal)}장")
print("=" * 60)

셀 4: 여러 Threshold로 평가 함수

In [ ]:
# ============================================
# Threshold 평가 함수
# ============================================

def evaluate_threshold(model, test_fire, test_normal, conf_threshold):
    """
    주어진 confidence threshold로 모델 평가
    
    Args:
        model: YOLO 모델
        test_fire: 화재 이미지 리스트
        test_normal: 정상 이미지 리스트
        conf_threshold: Confidence threshold 값
        
    Returns:
        dict: 평가 결과 (TP, FP, TN, FN, Recall, Precision, F1)
    """
    
    # 카운터 초기화
    tp = 0  # True Positive (화재를 화재로)
    fn = 0  # False Negative (화재를 못 찾음)
    tn = 0  # True Negative (정상을 정상으로)
    fp = 0  # False Positive (정상을 화재로)
    
    # ============================================
    # 화재 이미지 평가
    # ============================================
    
    for img_path in test_fire:
        # 예측 실행
        results = model.predict(
            str(img_path),
            conf=conf_threshold,
            imgsz=640,
            save=False,
            verbose=False
        )
        
        # bbox 개수 확인
        num_boxes = len(results[0].boxes)
        
        if num_boxes > 0:
            tp += 1  # 탐지 성공
        else:
            fn += 1  # 미탐
    
    # ============================================
    # 정상 이미지 평가
    # ============================================
    
    for img_path in test_normal:
        # 예측 실행
        results = model.predict(
            str(img_path),
            conf=conf_threshold,
            imgsz=640,
            save=False,
            verbose=False
        )
        
        # bbox 개수 확인
        num_boxes = len(results[0].boxes)
        
        if num_boxes > 0:
            fp += 1  # 오탐
        else:
            tn += 1  # 정상 맞춤
    
    # ============================================
    # 성능 지표 계산
    # ============================================
    
    # Recall = TP / (TP + FN)
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    # Precision = TP / (TP + FP)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    
    # F1 Score
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    # 결과 반환
    return {
        'threshold': conf_threshold,
        'tp': tp,
        'fp': fp,
        'tn': tn,
        'fn': fn,
        'recall': recall,
        'precision': precision,
        'f1': f1
    }

print("✅ 평가 함수 정의 완료")

셀 5: 여러 Threshold 테스트

In [ ]:
# ============================================
# 여러 Threshold 테스트
# ============================================

print("\n[Threshold 최적화 시작]")
print("=" * 60)

# 테스트할 Threshold 값들
thresholds = [0.25, 0.20, 0.15, 0.10, 0.05]

# 결과 저장 리스트
results_list = []

# 각 Threshold 평가
for threshold in thresholds:
    print(f"\n테스트 중: Threshold = {threshold}")
    
    # 평가 실행
    result = evaluate_threshold(model, test_fire, test_normal, threshold)
    
    # 결과 저장
    results_list.append(result)
    
    # 결과 출력
    print(f"  Recall:    {result['recall']:.3f}")
    print(f"  Precision: {result['precision']:.3f}")
    print(f"  F1:        {result['f1']:.3f}")
    print(f"  미탐:      {result['fn']}건")
    print(f"  오탐:      {result['fp']}건")
    
    # 목표 달성 확인
    if result['recall'] >= 0.920:
        print(f"  ✅ Recall 목표 달성!")
    else:
        print(f"  ⚠️ Recall {(0.920 - result['recall'])*100:.1f}%p 부족")

print("\n" + "=" * 60)
print("✅ 모든 Threshold 평가 완료")
print("=" * 60)

셀 6: 결과 시각화

In [ ]:
# ============================================
# 결과 시각화
# ============================================

print("\n[결과 시각화]")

# DataFrame 생성
df_results = pd.DataFrame(results_list)

# 그래프 생성
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Recall vs Threshold
axes[0, 0].plot(df_results['threshold'], df_results['recall'], 
                marker='o', linewidth=2, markersize=8, color='blue')
axes[0, 0].axhline(y=0.920, color='red', linestyle='--', label='목표 (0.920)')
axes[0, 0].set_xlabel('Confidence Threshold', fontsize=12)
axes[0, 0].set_ylabel('Recall', fontsize=12)
axes[0, 0].set_title('Recall vs Threshold', fontsize=14, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()
axes[0, 0].invert_xaxis()  # Threshold 작은 값이 오른쪽

# 2. Precision vs Threshold
axes[0, 1].plot(df_results['threshold'], df_results['precision'], 
                marker='s', linewidth=2, markersize=8, color='green')
axes[0, 1].axhline(y=0.880, color='red', linestyle='--', label='목표 (0.880)')
axes[0, 1].set_xlabel('Confidence Threshold', fontsize=12)
axes[0, 1].set_ylabel('Precision', fontsize=12)
axes[0, 1].set_title('Precision vs Threshold', fontsize=14, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()
axes[0, 1].invert_xaxis()

# 3. F1 Score vs Threshold
axes[1, 0].plot(df_results['threshold'], df_results['f1'], 
                marker='^', linewidth=2, markersize=8, color='purple')
axes[1, 0].set_xlabel('Confidence Threshold', fontsize=12)
axes[1, 0].set_ylabel('F1 Score', fontsize=12)
axes[1, 0].set_title('F1 Score vs Threshold', fontsize=14, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].invert_xaxis()

# 4. 미탐/오탐 vs Threshold
axes[1, 1].plot(df_results['threshold'], df_results['fn'], 
                marker='o', linewidth=2, markersize=8, color='red', label='미탐 (FN)')
axes[1, 1].plot(df_results['threshold'], df_results['fp'], 
                marker='s', linewidth=2, markersize=8, color='orange', label='오탐 (FP)')
axes[1, 1].set_xlabel('Confidence Threshold', fontsize=12)
axes[1, 1].set_ylabel('건수', fontsize=12)
axes[1, 1].set_title('오탐/미탐 vs Threshold', fontsize=14, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()
axes[1, 1].invert_xaxis()

plt.tight_layout()
plt.savefig(EVAL_DIR / 'threshold_optimization.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ 그래프 저장: {EVAL_DIR / 'threshold_optimization.png'}")

셀 7: 최적 Threshold 선정 및 결과 저장

In [ ]:
# ============================================
# 최적 Threshold 선정
# ============================================

print("\n" + "=" * 60)
print("📊 최적 Threshold 선정")
print("=" * 60)

# Recall 기준 정렬 (내림차순)
df_sorted = df_results.sort_values('recall', ascending=False)

print("\n【Recall 순위】")
for idx, row in df_sorted.iterrows():
    status = "✅" if row['recall'] >= 0.920 else "⚠️"
    print(f"  {row['threshold']:.2f}: Recall {row['recall']:.3f}, "
          f"Precision {row['precision']:.3f}, 미탐 {row['fn']}건, 오탐 {row['fp']}건 {status}")

# Recall 0.92 이상인 것 중 Precision 가장 높은 것 선택
candidates = df_results[df_results['recall'] >= 0.920]

if len(candidates) > 0:
    # Recall 목표 달성한 것 중 Precision 최대
    best = candidates.loc[candidates['precision'].idxmax()]
    print(f"\n✅ 목표 달성! 최적 Threshold: {best['threshold']:.2f}")
else:
    # 목표 미달성 시 Recall 최대
    best = df_results.loc[df_results['recall'].idxmax()]
    print(f"\n⚠️ 목표 미달성. Recall 최대 Threshold: {best['threshold']:.2f}")

print(f"\n【최적 설정】")
print(f"  Threshold:  {best['threshold']:.2f}")
print(f"  Recall:     {best['recall']:.3f} (목표: 0.920)")
print(f"  Precision:  {best['precision']:.3f} (목표: 0.880)")
print(f"  F1 Score:   {best['f1']:.3f}")
print(f"  미탐:       {best['fn']}건")
print(f"  오탐:       {best['fp']}건")

# ============================================
# 결과 저장
# ============================================

# CSV 저장
csv_path = EVAL_DIR / 'threshold_results.csv'
df_results.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f"\n✅ CSV 저장: {csv_path}")

# JSON 저장
json_data = {
    'optimization_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    'model': 'YOLOv8n',
    'thresholds_tested': thresholds,
    'optimal_threshold': float(best['threshold']),
    'optimal_performance': {
        'recall': float(best['recall']),
        'precision': float(best['precision']),
        'f1_score': float(best['f1']),
        'false_negative': int(best['fn']),
        'false_positive': int(best['fp'])
    },
    'all_results': results_list
}

json_path = EVAL_DIR / 'optimization_results.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(json_data, f, indent=2, ensure_ascii=False)

print(f"✅ JSON 저장: {json_path}")

print("\n" + "=" * 60)
print("🎉 Threshold 최적화 완료!")
print("=" * 60)
```

---

## 🎯 예상 결과
```
Threshold 0.25: Recall 약 0.89
Threshold 0.20: Recall 0.915 (현재)
Threshold 0.15: Recall 약 0.93-0.94 ✅
Threshold 0.10: Recall 약 0.95-0.96
Threshold 0.05: Recall 약 0.97-0.98

→ 0.15가 최적일 가능성 높음!